# MedTranslate Guard: can we trust machine translation of medicine instructions?

**Veritas AI International High School Hackathon 2026 · AI + Languages track**

**Team:** Wabi Tafese, Natan Tilahun and Tamerat Akililu

**One-sentence pitch:** MedTranslate Guard takes an English medicine instruction, translates it into Amharic or Afaan Oromo, and warns the user, with a readable reason, when the translation might be dangerous.

**How to run:** Runtime → Change runtime type → GPU (recommended), then Runtime → Run all. The first run downloads the translation model, so it takes a few minutes. It needs no API keys or uploads.

*This project is a prototype and is not medical advice. "No red flag" never means "safe". Always confirm medicine instructions with a pharmacist or health worker.*

## 1. Get the code and data

Everything (code, sentences, translations, human ratings) is in our GitHub repository. This cell downloads it and installs the libraries.

In [ ]:
!git clone -q https://github.com/wabix2/medtranslate-guard.git
%cd medtranslate-guard
!pip install -q -r requirements.txt

## 2. The data

We wrote **200 short English medicine instructions** in two rounds (S01-S100 and S101-S200), in four categories: frequency/duration, negation/warnings, quantity, and route/condition. No external datasets and no real patient data were used.

In [ ]:
import pandas as pd
df = pd.read_csv("sentences.csv")
print(df.shape)
df.head(10)

## 3. Try it yourself

`check_instruction` translates the sentence with NLLB-200 (distilled 600M), translates it back to English, and runs our three-layer safety gate. Change the sentence or the language (`"amharic"` or `"oromo"`) and run it again.

In [ ]:
import medtranslate_guard as mg

examples = [
    ("Take one lozenge every two hours.", "amharic"),
    ("Do not take this medicine with alcohol.", "amharic"),
    ("Store in a cool, dry place.", "oromo"),
]
for text, lang in examples:
    print("=" * 70)
    print(text, "|", lang)
    print(mg.check_instruction(text, lang=lang))

## 4. How the safety gate works

We wanted warnings a person can understand, so the gate is rule-based rather than a black box. It has three layers:

1. **Checks on the translation itself** (for example, a suspicious length or missing numbers).
2. **Checks after back-translation** (does the English that comes back still mean the same thing?), using sentence-transformers `all-MiniLM-L6-v2` for meaning similarity.
3. **Lost units, conditions and key terms** (such as "not", "hours", "with food").

Every warning comes with a readable reason. The thresholds were chosen on Round 1 only and locked in `locked_thresholds.json`, so Round 2 is a fair test.

## 5. Results on the held-out Round 2 set

Our baseline was chrF only (the standard translation score), set to the same recall as our gate. Our gate gave fewer false warnings.

| Round 2 (held out) | Amharic | Afaan Oromo |
|---|---|---|
| Dangerous translations (human rating) | 31 of 100 | 27 of 100 |
| Caught by the gate (95% CI) | 25 = 81% (64-91%) | 25 = 93% (77-98%) |
| Dangerous missed by the gate | 6 | 2 |
| False warnings: gate / chrF-only | 30 / 36 | 37 / 46 |
| Sentences flagged | 55% | 62% |
| chrF alone detects dangerous (AUROC) | 0.70 | 0.77 |

The next cell re-scores Round 2 from the saved translations and ratings (the thresholds stay locked), so you can check the numbers yourself.

In [ ]:
import medtranslate_guard as mg
R2 = "round2"
mg.SENTENCES_FILE = f"{R2}/sentences.csv"
mg.SCORED_FILE    = f"{R2}/translations_scored.csv"
mg.RATED_FILE     = f"{R2}/translations_rated.csv"
mg.SHEET_DIR      = f"{R2}/rating_sheets"
mg.FIG_DIR        = f"{R2}/figures"
mg.PROVENANCE_FILE= f"{R2}/provenance.json"
mg.SUMMARY_FILE   = f"{R2}/results_summary.md"
mg.RETEST_N = 0
_orig = mg.add_chrf
mg.add_chrf = lambda df, verbose=False: _orig(df, verbose=False)
mg.main()

In [ ]:
import glob
from IPython.display import Image, display
print(open("round2/results_summary.md").read())
for f in sorted(glob.glob("round2/figures/*.png")):
    print(f)
    display(Image(f))

## 6. Where it fails, and why

- **chrF is only a moderate signal** (AUROC 0.70-0.77). A dangerous translation can still score a middling chrF: Oromo S196, "ten seconds" became "ten hours", chrF 54.
- **The gate warns on more than half of all sentences**, so many warnings are false alarms. Its main benefit over chrF is that each warning explains itself.
- **Quantity and route/condition instructions are the riskiest** (33-45% dangerous). The gate missed "cool, dry place" becoming "warm, warm place" in Oromo. We think this is because back-translation uses the same model, so a consistent error survives the round trip.

**Limitations:** the test set is deliberately hard, so these are not real-world error rates. Oromo has one rater, who is also on our team. Amharic rater agreement fell in Round 2 (weighted kappa 0.41 against 0.73 in Round 1). chrF uses one reference per sentence.

## 7. What we'd do next

1. Get more independent Oromo and Amharic raters (ideally health workers or pharmacists) to make the ratings more reliable.
2. Test the gate on real medicine leaflets and use a second, different translation model for the back-translation check, to cut false warnings and catch consistent errors.

## 8. Sources and AI tools

- NLLB-200 distilled 600M (Meta AI): https://huggingface.co/facebook/nllb-200-distilled-600M
- sentence-transformers `all-MiniLM-L6-v2`: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- sacreBLEU (chrF): https://github.com/mjpost/sacrebleu
- scikit-learn: https://scikit-learn.org
- Code, data and ratings: https://github.com/wabix2/medtranslate-guard
- All sentences, reference translations and safety ratings were created by our team.
- AI tools used: **Claude (Anthropic) for debugging, writing code and the write-up; ChatGPT (OpenAI) for brainstorming ideas and analysis**